# NASA EONET Data At 18:00, 30-06-2026

Extracting data from EONET

In [1]:
import requests

url = "https://eonet.gsfc.nasa.gov/api/v3/events"

response = requests.get(url)

print(response.status_code)

data = response.json()


print(data)

503
{'message': 'Service is experiencing high demand. Please try again in a moment.', 'retry_after': 3}


Saving response in data/nasa/data.py for structural review

In [ ]:
import requests
from pprint import pformat

url = "https://eonet.gsfc.nasa.gov/api/v3/events"

response = requests.get(url)

data = response.json()

with open("data/nasa/data.py", "w", encoding="utf-8") as f:
    f.write("# Auto-generated from NASA EONET API\n\n")
    f.write("DATA = ")
    f.write(pformat(data, width=120))

In [ ]:
import pandas as pd

events = data["events"]

Exploring categories

In [ ]:
categories_rows = []

for event in events:
    for category in event["categories"]:
        categories_rows.append({
            "category_id": category["id"],
            "category_title": category["title"]
        })

categories_df = pd.DataFrame(categories_rows)

In [7]:
categories_df

,category_id,category_title
0,seaLakeIce,Sea and Lake Ice
1,severeStorms,Severe Storms
2,severeStorms,Severe Storms
3,wildfires,Wildfires
4,wildfires,Wildfires
...,...,...
6734,seaLakeIce,Sea and Lake Ice
6735,seaLakeIce,Sea and Lake Ice
6736,seaLakeIce,Sea and Lake Ice
6737,seaLakeIce,Sea and Lake Ice


In [8]:
categories_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6739 entries, 0 to 6738
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   category_id     6739 non-null   str  
 1   category_title  6739 non-null   str  
dtypes: str(2)
memory usage: 224.2 KB


In [9]:
categories_df[categories_df.duplicated()]

,category_id,category_title
2,severeStorms,Severe Storms
4,wildfires,Wildfires
5,wildfires,Wildfires
6,wildfires,Wildfires
7,wildfires,Wildfires
...,...,...
6734,seaLakeIce,Sea and Lake Ice
6735,seaLakeIce,Sea and Lake Ice
6736,seaLakeIce,Sea and Lake Ice
6737,seaLakeIce,Sea and Lake Ice


In [10]:
categories_df = categories_df.drop_duplicates()

In [11]:
categories_df

,category_id,category_title
0,seaLakeIce,Sea and Lake Ice
1,severeStorms,Severe Storms
3,wildfires,Wildfires
14,volcanoes,Volcanoes


In [12]:
categories_df = categories_df.set_index("category_id")

In [13]:
categories_df

,category_title
category_id,
seaLakeIce,Sea and Lake Ice
severeStorms,Severe Storms
wildfires,Wildfires
volcanoes,Volcanoes


In [42]:
categories_df.to_csv("data/nasa/categories.csv")

Exploring sources

In [ ]:
sources_rows = []

for event in events:
    for source in event["sources"]:
        sources_rows.append({
            "event_id": event["id"],
            "source_id": source["id"],
            "source_title": source["url"]
        })

sources_df = pd.DataFrame(sources_rows)

In [21]:
sources_df

,event_id,source_id,source_title
0,EONET_20822,NATICE,https://usicecenter.gov/pub/Iceberg_Tabular.csv
1,EONET_20671,JTWC,https://www.metoc.navy.mil/jtwc/products/wp082...
2,EONET_20606,JTWC,https://www.metoc.navy.mil/jtwc/products/wp072...
3,EONET_20558,IRWIN,https://irwin.doi.gov/observer/incidents/5c3a9...
4,EONET_20565,IRWIN,https://irwin.doi.gov/observer/incidents/8d86b...
...,...,...,...
6755,EONET_980,GDACS,http://www.gdacs.org/report.aspx?eventtype=VO&...
6756,EONET_980,IDC,https://disasterscharter.org/web/guest/-/volca...
6757,EONET_980,NASA_DISP,https://disasters.nasa.gov/mount-fuego-eruptio...
6758,EONET_980,ReliefWeb,https://reliefweb.int/disaster/vo-2018-000066-gtm


In [41]:
sources_df.to_csv("data/nasa/sources.csv", index=False)

Exploring geometries

In [ ]:
geometry_rows = []

for event in events:
    for geo in event["geometry"]:
        geometry_rows.append({
            "event_id": event["id"],
            "date": geo["date"],
            "type": geo["type"],
            "longitude": geo["coordinates"][0],
            "latitude": geo["coordinates"][1],
            "magnitude": geo.get("magnitudeValue"),
            "magnitude_unit": geo.get("magnitudeUnit")
        })

Unique uuid for geo locations

In [34]:
from uuid import uuid4

geometry_df = pd.DataFrame(geometry_rows)
geometry_df["geometry_id"] = ["GEO_" + uuid4().hex[:10].upper() for _ in range(len(geometry_df))]

In [35]:
geometry_df

,event_id,date,type,longitude,latitude,magnitude,magnitude_unit,geometry_id
0,EONET_20822,2026-06-25T00:00:00Z,Point,-55.5100,-64.3500,62.73,NM^2,GEO_8C4D06FABA
1,EONET_20671,2026-06-23T00:00:00Z,Point,145.7000,14.7000,35.00,kts,GEO_F9A5348968
2,EONET_20671,2026-06-23T06:00:00Z,Point,144.8000,14.7000,35.00,kts,GEO_B80DBF3AED
3,EONET_20671,2026-06-23T12:00:00Z,Point,143.9000,14.8000,30.00,kts,GEO_6CD53B22A9
4,EONET_20671,2026-06-23T18:00:00Z,Point,142.8000,14.9000,30.00,kts,GEO_85015C2AE4
...,...,...,...,...,...,...,...,...
7309,EONET_2878,2011-08-30T00:00:00Z,Point,143.3638,-66.3660,NaN,NaN,GEO_0861F15D79
7310,EONET_2878,2013-04-08T00:00:00Z,Point,142.6649,-66.7354,NaN,NaN,GEO_3CD9ACC539
7311,EONET_2878,2018-01-12T00:00:00Z,Point,143.3900,-66.3700,NaN,NaN,GEO_3C390DBF1F
7312,EONET_2997,2011-08-30T00:00:00Z,Point,96.1800,-64.9955,NaN,NaN,GEO_D0B7DA8FB8


In [36]:
geometry_df = geometry_df.set_index("geometry_id")

In [37]:
geometry_df

,event_id,date,type,longitude,latitude,magnitude,magnitude_unit
geometry_id,,,,,,,
GEO_8C4D06FABA,EONET_20822,2026-06-25T00:00:00Z,Point,-55.5100,-64.3500,62.73,NM^2
GEO_F9A5348968,EONET_20671,2026-06-23T00:00:00Z,Point,145.7000,14.7000,35.00,kts
GEO_B80DBF3AED,EONET_20671,2026-06-23T06:00:00Z,Point,144.8000,14.7000,35.00,kts
GEO_6CD53B22A9,EONET_20671,2026-06-23T12:00:00Z,Point,143.9000,14.8000,30.00,kts
GEO_85015C2AE4,EONET_20671,2026-06-23T18:00:00Z,Point,142.8000,14.9000,30.00,kts
...,...,...,...,...,...,...,...
GEO_0861F15D79,EONET_2878,2011-08-30T00:00:00Z,Point,143.3638,-66.3660,NaN,NaN
GEO_3CD9ACC539,EONET_2878,2013-04-08T00:00:00Z,Point,142.6649,-66.7354,NaN,NaN
GEO_3C390DBF1F,EONET_2878,2018-01-12T00:00:00Z,Point,143.3900,-66.3700,NaN,NaN


In [40]:
geometry_df.to_csv("data/nasa/geometry.csv")

Exploring events

In [ ]:
events_rows = []

for event in events:
    events_rows.append({
        "event_id": event["id"],
        "category_id": event["categories"][0]["id"],
        "title": event["title"],
        "description": event["description"],
        "closed": event["closed"],
        "link": event["link"]
    })

events_df = pd.DataFrame(events_rows)

In [46]:
events_df

,event_id,category_id,title,description,closed,link
0,EONET_20822,seaLakeIce,Iceberg D33D,NaN,None,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...
1,EONET_20671,severeStorms,Tropical Storm Higos,NaN,None,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...
2,EONET_20606,severeStorms,Typhoon Mekkhala,NaN,None,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...
3,EONET_20558,wildfires,"Median Wildfire, Gooding, Idaho",NaN,None,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...
4,EONET_20565,wildfires,"TRAIL 13 (09) Wildfire, Citrus, Florida","6 Miles SW from INVERNESS, FL",None,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...
...,...,...,...,...,...,...
6734,EONET_2736,seaLakeIce,Iceberg B22A,NaN,None,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...
6735,EONET_2876,seaLakeIce,Iceberg C15,NaN,None,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...
6736,EONET_2878,seaLakeIce,Iceberg B09B,NaN,None,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...
6737,EONET_2997,seaLakeIce,Iceberg C21B,NaN,None,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...


In [47]:
events_df = events_df.set_index("event_id")

In [48]:
events_df

,category_id,title,description,closed,link
event_id,,,,,
EONET_20822,seaLakeIce,Iceberg D33D,NaN,None,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...
EONET_20671,severeStorms,Tropical Storm Higos,NaN,None,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...
EONET_20606,severeStorms,Typhoon Mekkhala,NaN,None,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...
EONET_20558,wildfires,"Median Wildfire, Gooding, Idaho",NaN,None,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...
EONET_20565,wildfires,"TRAIL 13 (09) Wildfire, Citrus, Florida","6 Miles SW from INVERNESS, FL",None,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...
...,...,...,...,...,...
EONET_2736,seaLakeIce,Iceberg B22A,NaN,None,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...
EONET_2876,seaLakeIce,Iceberg C15,NaN,None,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...
EONET_2878,seaLakeIce,Iceberg B09B,NaN,None,https://eonet.gsfc.nasa.gov/api/v3/events/EONE...


In [49]:
events_df.info()

<class 'pandas.DataFrame'>
Index: 6739 entries, EONET_20822 to EONET_980
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   category_id  6739 non-null   str   
 1   title        6739 non-null   str   
 2   description  1124 non-null   str   
 3   closed       0 non-null      object
 4   link         6739 non-null   str   
dtypes: object(1), str(4)
memory usage: 1.3+ MB


In [50]:
events_df = events_df.drop(columns = ["description"])

In [54]:
events_df = events_df.replace({None: 0})

In [57]:
events_df["closed"] = events_df["closed"].astype(bool)

In [58]:
events_df.info()

<class 'pandas.DataFrame'>
Index: 6739 entries, EONET_20822 to EONET_980
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   category_id  6739 non-null   str  
 1   title        6739 non-null   str  
 2   closed       6739 non-null   bool 
 3   link         6739 non-null   str  
dtypes: bool(1), str(3)
memory usage: 1.2 MB


In [59]:
events_df.to_csv("data/nasa/events.csv")